# 此文档用于动手实现pytorch的一些基础功能
* 包括线性回归，sofatmax回归，MLP,CNN

## 1.线性回归的pytorch实现
* 不使用api的原始实现

In [66]:
import random
import torch as T
import matplotlib.pyplot as plt

# 数据生成器
def generate_data(w,b,num):
    X = T.normal(0,1,size=(num,len(w)))
    y = T.matmul(X,w) + b
    y += T.normal(0,0.01,size=(num,1))
    return X,y

# 数据输出器
def data_iter(batch_size,features,labels):
    num = len(features)
    index = [i for i in range(num)]
    random.shuffle(index) # 随机打乱数据
    for i in range(0,num,batch_size):
        batch_index = T.tensor(index[i:min(i + batch_size,num)])
        yield features[batch_index],labels[batch_index] #按batch_size输出

# 定义模型
def line(X,w,b):
    return T.matmul(X,w) + b

# 定义损失函数
def loss(y_hat,y):
    return ((y_hat-y)**2)/2

# 定义优化算法
def sgd (paras,lr,batch_size):
    with T.no_grad():
        for para in paras:
            para -= lr * para.grad/batch_size
            para.grad.zero_() # 用于清零当前参数的梯度，以便在下一次反向传播时重新计算

# 真实数据
true_w = T.tensor([[2],[-3.4]])
true_b = T.tensor(4.2)

features,labels = generate_data(true_w,true_b,1000)
# plt只能画数组型的数据，需要使用.numpy()转化
# plt.scatter(features[:,1].numpy(),labels.numpy(),1)

batch_size = 10 # batch大小
w = T.zeros(size=(2,1),requires_grad=True)
b = T.zeros(1,requires_grad=True)

# 训练
lr = 0.01
epoch = 5

for i in range(epoch):
    for X,y in data_iter(batch_size,features,labels):
        l = loss(line(X,w,b),y)
        l.sum().backward()
        sgd([w,b],lr,batch_size)
    with T.no_grad():
        tran_l = loss(line(features,w,b),labels)
        print("epoch:",i + 1,"\n","loss:",tran_l.mean().numpy())

print("w:",w,"\n","b:",b)

epoch: 1 
 loss: 2.1297317
epoch: 2 
 loss: 0.26636225
epoch: 3 
 loss: 0.03388089
epoch: 4 
 loss: 0.0044209873
epoch: 5 
 loss: 0.0006264636
w: tensor([[ 1.9841],
        [-3.3899]], requires_grad=True) 
 b: tensor([4.1712], requires_grad=True)


* 简洁实现版本

In [23]:
import numpy as np
import torch as T
from torch.utils import data
from torch import nn

# 数据生成器
def generate_data(w,b,num):
    X = T.normal(0,1,size=(num,len(w)))
    y = T.matmul(X,w) + b
    y += T.normal(0,0.01,size=(num,1))
    return X,y

# 读取数据集(这里使用高级api实现)
def load_array(features,labels,batch_size):
    data_set = data.TensorDataset(features,labels)
    return data.DataLoader(data_set,batch_size=batch_size,shuffle=True)

# 真实数据
true_w = T.tensor([[2],[-3.4]])
true_b = T.tensor(4.2)

# 数据生成器定义
features,labels = generate_data(true_w,true_b,1000)
data_iter = load_array(features,labels,10)

# 网络定义
net = nn.Sequential(nn.Linear(2,1))
# 参数初始化
net[0].weight.data.normal_(0,0.01)
net[0].bias.data.fill_(0)

# 定义损失函数
loss = nn.MSELoss()

# 定义优化算法
trainer = T.optim.SGD(net.parameters(),lr=0.03)

# 训练
epochs = 3
for epoch in range(epochs):
    for X,y in data_iter:
        l = loss(net(X),y)
        # 以下三步可以说是封装的非常简洁了
        trainer.zero_grad()
        l.backward()
        trainer.step()
    # with这句话非常关键，不然会Can't call numpy() on Tensor that requires grad.
    # with T.no_grad():
    tran_l = loss(net(features),labels)
    print(net[0].weight.data)
    print("epoch:",epoch + 1,"\n","loss:",tran_l.detach().numpy())
    


tensor([[ 1.9949, -3.3891]])
epoch: 1 
 loss: 0.00036634403
tensor([[ 1.9994, -3.3982]])
epoch: 2 
 loss: 9.3740746e-05
tensor([[ 2.0003, -3.3991]])
epoch: 3 
 loss: 9.246109e-05


## 2.softmax回归的pytorch实现
* 原始实现
（交叉熵是度量两个概率分布差异的很好的度量，而平方差损失度量的是两个样本之间的差异）

In [ ]:
import torch as T
import torchvision as tv
from torch.utils import data
from torchvision import transforms

# 定义数据生成器
def load_data(batch_size):
    # 将图像从 PIL 格式或 NumPy 数组转换为 PyTorch 的张量（tensor）
    trans = [transforms.ToTensor()]
    # 是一个用于将多个转换操作组合在一起的工具
    trans = transforms.Compose(trans)
    # 加载mnist数据集
    train_data = tv.datasets.MNIST(root='./data',train=True,download=True,transform=trans)
    test_data = tv.datasets.MNIST(root='./data',train=False,download=True,transform=trans)
    # 定义数据加载器
    train_iter = data.DataLoader(train_data,batch_size=batch_size,shuffle=True)
    test_iter = data.DataLoader(test_data,batch_size=batch_size,shuffle=False)
    return train_iter,test_iter

batch_size = 256
train_iter,test_iter = load_data(batch_size)

# 确定输入和输出
num_inputs = 784
num_outputs = 10

w = T.randn(num_inputs,num_outputs,requires_grad=True)
b = T.zeros(num_outputs,requires_grad=True)

# 定义softmax函数
def softmax(X):
    X_exp = T.exp(X)
    partition = X_exp.sum(dim=1,keepdim=True)
    return X_exp / partition # 广播机制，很好用

 # 定义模型
def net(X):
    return softmax(T.matmul(X.reshape(-1,num_inputs),w) + b)

# 定义损失函数
def cross_entropy(y_hat,y):
    return -T.log(y_hat[T.arange(len(y_hat)),y])

torch.Size([784, 10])
torch.Size([10])
